# Degree-Aware Curriculum Hard-Negative Sampling for LightGCL

**Companion Colab notebook** for the class project report *"Degree-Aware Curriculum
Hard-Negative Sampling for LightGCL"* (Graph Analytics for Big Data, IT5429E).

**Research question.** Does a hard-negative curriculum modulated by user activity
improve LightGCL's recommendation accuracy and long-tail robustness, without
increasing false-negative bias?

**What this notebook does.** It reproduces the core mechanism end-to-end on the real
Yelp2018 benchmark, at reduced scale (fewer epochs, single seed) so it finishes in a
few minutes on a free Colab GPU:
1. Loads the real Yelp2018 train/test interaction matrices.
2. Computes Personalized PageRank (PPR) hard-negative candidates per user.
3. Trains LightGCL under the **S0 (uniform)** and **S3 (degree-gated curriculum)**
   negative samplers side by side.
4. Evaluates Recall@20 / NDCG@20 under full ranking and plots the comparison.

**What this notebook is *not*.** A replacement for the paper's multi-seed, 100-epoch
results. Those (all four samplers S0-S3, 3 baselines, degree-bucket breakdown, full
hyperparameter sweep) are in `results/*.csv` and `report/main.pdf` in the project
repository; this notebook is a fast, inspectable, correctness-oriented demo of the
same code, not a substitute for that evaluation.

Runtime: ~5-10 minutes on a free Colab T4 GPU with the default settings below.

## 1. Setup

Clones the project repository (which ships the Yelp2018 `trnMat.pkl` / `tstMat.pkl`
splits used throughout the report, so no separate dataset download is needed) and
installs the small set of missing dependencies.

In [ ]:
import os

REPO_URL = 'https://github.com/dang-nh/GraphML-LightGCN.git'
REPO_DIR = 'GraphML-LightGCN'

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL}
%cd {REPO_DIR}

!pip install -q torch numpy scipy pandas tqdm matplotlib

In [ ]:
import sys
sys.path.insert(0, 'third_party/LightGCL')  # for `import model, utils` (vendored LightGCL)

import pickle
import time

import numpy as np
import pandas as pd
import torch
import torch.utils.data as tdata
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from model import LightGCL
from utils import TrnData, metrics, scipy_sparse_mat_to_torch_sparse_tensor

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)
torch.manual_seed(0)
np.random.seed(0)

## 2. Data loading

Yelp2018 is an implicit-feedback benchmark of user check-ins at local businesses. We
use the exact train/test split shipped with the official LightGCL repository (see
`report/main.tex`, Appendix A, for a note on why these statistics differ slightly from
the project proposal draft).

In [ ]:
def load_yelp2018(data_dir='third_party/LightGCL/data/yelp'):
    """Load the Yelp2018 train/test implicit-feedback matrices used throughout this
    project (the exact split shipped with the official LightGCL repository).

    Args:
        data_dir: directory containing ``trnMat.pkl`` and ``tstMat.pkl``, each a
            ``scipy.sparse`` (n_user, n_item) matrix of implicit interactions.

    Returns:
        train_coo: training interactions as a COO matrix (used for propagation, PPR,
            and BPR negative sampling).
        train_csr: training interactions as a CSR matrix (used to mask seen items at
            test time).
        test_labels: list indexed by user id of held-out test item ids.
    """
    with open(f'{data_dir}/trnMat.pkl', 'rb') as f:
        train = pickle.load(f)
    with open(f'{data_dir}/tstMat.pkl', 'rb') as f:
        test = pickle.load(f)
    train_csr = (train != 0).astype(np.float32)
    train_coo = train.tocoo()
    test = test.tocoo()
    test_labels = [[] for _ in range(test.shape[0])]
    for r, c in zip(test.row, test.col):
        test_labels[r].append(c)
    return train_coo, train_csr, test_labels


train_coo, train_csr, test_labels = load_yelp2018()
n_user, n_item = train_coo.shape
n_test_interactions = sum(len(x) for x in test_labels)
# density is computed over train+test combined, matching report/main.tex Table 1
density = 100 * (train_coo.nnz + n_test_interactions) / (n_user * n_item)
print(f'Yelp2018: {n_user:,} users, {n_item:,} items, '
      f'{train_coo.nnz:,} train interactions, {n_test_interactions:,} test interactions, '
      f'density {density:.3f}%')

## 3. Personalized PageRank (PPR) hard-negative candidates

For each user we rank all non-interacted items by Personalized PageRank on the
symmetric-normalized user-item bipartite graph and keep a percentile band
$[p_{lo}, p_{hi}]$ of that ranking as the hard-negative candidate pool (following
PinSAGE's PPR-based hard-negative construction). This is a **one-time, per-dataset**
precompute (11.4s for Yelp2018 on an RTX 5090 in the full paper run); it does not
repeat during training. See `scripts/precompute_ppr.py` and
`docs/design/sampler_design.md` §1-2 for the full, GPU-batched version used for the
paper's results; the function below is the same algorithm, simplified for a single
notebook cell.

In [ ]:
def build_symmetric_adj(train_coo, n_user, n_item, device):
    """Symmetric-normalized bipartite adjacency D^-1/2 A D^-1/2 as an (N, N) sparse
    tensor, N = n_user + n_item, stacked as [[0, R], [R^T, 0]]. Also returns each
    user's raw training degree (used by the degree gate below)."""
    rowD = np.array(train_coo.sum(1)).squeeze()
    colD = np.array(train_coo.sum(0)).squeeze()
    rows, cols = train_coo.row, train_coo.col
    vals = 1.0 / np.sqrt(np.maximum(rowD[rows], 1) * np.maximum(colD[cols], 1))

    N = n_user + n_item
    r_rows, r_cols = rows, cols + n_user
    t_rows, t_cols = cols + n_user, rows
    all_rows = np.concatenate([r_rows, t_rows])
    all_cols = np.concatenate([r_cols, t_cols])
    all_vals = np.concatenate([vals, vals]).astype(np.float32)

    indices = torch.from_numpy(np.vstack([all_rows, all_cols]).astype(np.int64))
    values = torch.from_numpy(all_vals)
    adj = torch.sparse_coo_tensor(indices, values, (N, N)).coalesce().to(device)
    return adj, rowD.astype(np.int32)


def compute_ppr_pool(train_coo, n_user, n_item, device, c=0.15, n_iter=30,
                      p_lo=0.02, p_hi=0.10, pool_size=500, chunk=4096):
    """GPU-batched Personalized PageRank hard-negative pool, one-time per dataset.

    For each user, solves the PPR fixed point pi_u = (1-c) A~ pi_u + c e_u by power
    iteration, ranks items by score (excluding observed positives), and keeps a
    fixed-size uniform sample from the [p_lo, p_hi] percentile band of that ranking
    as the hard-negative candidate pool (report §4.3, Algorithm 1).

    Returns:
        ppr_pool: (n_user, pool_size) int32 array of hard-negative item candidates.
        deg: (n_user,) int32 array of training degrees (for the degree gate).
    """
    adj, deg = build_symmetric_adj(train_coo, n_user, n_item, device)
    N = n_user + n_item
    train_csr_local = train_coo.tocsr()
    pool = np.zeros((n_user, pool_size), dtype=np.int32)
    rng = np.random.default_rng(0)

    n_chunks = int(np.ceil(n_user / chunk))
    for ci in tqdm(range(n_chunks), desc='PPR precompute'):
        u0, u1 = ci * chunk, min((ci + 1) * chunk, n_user)
        cs = u1 - u0

        E = torch.zeros(N, cs, device=device)
        E[torch.arange(u0, u1, device=device), torch.arange(cs, device=device)] = 1.0
        X = E.clone()
        for _ in range(n_iter):
            X = (1 - c) * torch.sparse.mm(adj, X) + c * E

        item_scores = X[n_user:N, :].transpose(0, 1).contiguous()
        mask = torch.from_numpy(train_csr_local[u0:u1].toarray()).to(device).bool()
        item_scores = item_scores.masked_fill(mask, float('-inf'))

        k_hi = int(np.ceil(p_hi * n_item))
        top_vals, top_idx = torch.topk(item_scores, k=k_hi, dim=1)
        sort_ord = torch.argsort(top_vals, dim=1, descending=True)
        top_idx_sorted = torch.gather(top_idx, 1, sort_ord).cpu().numpy()

        lo = int(np.floor(p_lo * n_item))
        band_items = top_idx_sorted[:, lo:k_hi]
        for i in range(cs):
            pool[u0 + i] = rng.choice(band_items[i], size=pool_size, replace=True)

    return pool, deg


t0 = time.time()
ppr_pool, deg = compute_ppr_pool(train_coo, n_user, n_item, DEVICE)
print(f'PPR pool computed in {time.time() - t0:.1f}s, shape {ppr_pool.shape}')

## 4. Model: LightGCN propagation + LightGCL SVD-contrastive view

We reuse the project's `LightGCL` model class (`third_party/LightGCL/model.py`,
docstring included there) unmodified: LightGCN message passing over the normalized
adjacency provides the ranking embeddings, contrasted via InfoNCE against a
truncated-SVD global view. Only the BPR negative sampler (next section) changes
between the S0 and S3 runs below; the model and its contrastive branch are identical.

In [ ]:
def build_model(train_coo, n_user, n_item, device, d=64, l=2, svd_q=5, temp=0.2,
                 lambda_1=0.2, lambda_2=1e-7, dropout=0.0, batch_user=256):
    """Instantiate a fresh LightGCL model and its normalized-adjacency / SVD inputs.

    Args:
        train_coo: training interactions (COO), symmetric-normalized in place here
            (matches the official LightGCL preprocessing).
        d: embedding dimension. l: number of GNN propagation layers.
        svd_q: truncated-SVD rank for the contrastive view.
        temp, lambda_1, lambda_2, dropout, batch_user: LightGCL hyperparameters,
            left at the repository's published defaults (not re-tuned in this demo).

    Returns:
        (model, adj_norm): the LightGCL module (already moved to `device`) and its
        normalized adjacency tensor.
    """
    rowD = np.array(train_coo.sum(1)).squeeze()
    colD = np.array(train_coo.sum(0)).squeeze()
    norm_coo = train_coo.copy()
    for i in range(len(norm_coo.data)):
        norm_coo.data[i] /= pow(rowD[norm_coo.row[i]] * colD[norm_coo.col[i]], 0.5)

    adj_norm = scipy_sparse_mat_to_torch_sparse_tensor(norm_coo).coalesce().to(device)
    svd_u, s, svd_v = torch.svd_lowrank(adj_norm, q=svd_q)
    u_mul_s, v_mul_s = svd_u @ torch.diag(s), svd_v @ torch.diag(s)

    model = LightGCL(n_user, n_item, d, u_mul_s, v_mul_s, svd_u.T, svd_v.T,
                      train_csr, adj_norm, l, temp, lambda_1, lambda_2, dropout,
                      batch_user, device)
    return model.to(device), adj_norm

## 5. Training and evaluation loop

`TrnData` (`third_party/LightGCL/utils.py`) implements the four sampler modes
described in the report: **S0** uniform, **S1** pure PPR, **S2** fixed-ramp mixture,
**S3** our degree-gated curriculum mixture. We compare **S0 vs. S3** here (the paper's
central research-question comparison); the full S0-S3 ablation with 2-3 seeds is in
`results/ablation_summary.csv`.

In [ ]:
def evaluate(model, adj_norm, test_labels, device, topk=(20,), batch_user=256):
    """Full-ranking evaluation: rank all items for every user, mask seen training
    items, and compute Recall@k / NDCG@k for each k in `topk`.

    Returns a dict {f'recall@{k}': value, f'ndcg@{k}': value, ...}.
    """
    model.eval()
    n_user = adj_norm.shape[0]
    test_uids = np.arange(n_user)
    n_batches = int(np.ceil(n_user / batch_user))
    totals = {k: [0.0, 0.0] for k in topk}  # k -> [recall_sum, ndcg_sum]

    with torch.no_grad():
        for b in range(n_batches):
            start, end = b * batch_user, min((b + 1) * batch_user, n_user)
            uids = torch.LongTensor(test_uids[start:end]).to(device)
            preds = model(uids, None, None, None, test=True).cpu().numpy()
            for k in topk:
                r, n = metrics(test_uids[start:end], preds, k, test_labels)
                totals[k][0] += r
                totals[k][1] += n
    return {f'{name}@{k}': totals[k][i] / n_batches
            for k in topk for i, name in enumerate(['recall', 'ndcg'])}


def train_and_evaluate(neg_mode, train_coo, n_user, n_item, ppr_pool, deg, test_labels,
                        device, n_epochs=20, eval_every=4, inter_batch=4096,
                        lr=1e-3, seed=0, **model_kwargs):
    """Train LightGCL end-to-end under a given negative-sampler mode and track
    Recall@20/NDCG@20 over epochs.

    Args:
        neg_mode: one of 'S0' (uniform), 'S1' (pure PPR), 'S2' (fixed-ramp mixture),
            'S3' (degree-gated curriculum) -- see `utils.TrnData`.
        n_epochs: reduced from the paper's 100 for a fast Colab demo.
        eval_every: evaluate Recall@20/NDCG@20 every this many epochs.

    Returns:
        dict with 'loss' (per-epoch total training loss), 'eval_epochs',
        'recall20', 'ndcg20' (evaluated every `eval_every` epochs), and 'final'
        (the last evaluation's metric dict).
    """
    torch.manual_seed(seed)
    np.random.seed(seed)

    model, adj_norm = build_model(train_coo, n_user, n_item, device, **model_kwargs)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    train_data = TrnData(train_coo, neg_mode=neg_mode, ppr_pool=ppr_pool, deg=deg,
                          alpha_bar=0.3, Tw=10, gate_a=1.0, seed=seed)
    loader = tdata.DataLoader(train_data, batch_size=inter_batch, shuffle=True,
                               num_workers=0)

    history = {'loss': [], 'eval_epochs': [], 'recall20': [], 'ndcg20': []}
    for epoch in tqdm(range(n_epochs), desc=f'Training {neg_mode}'):
        model.train()
        loader.dataset.neg_sampling(epoch)  # one vectorized resample per epoch
        epoch_loss, n_batches = 0.0, 0
        for uids, pos, neg in loader:
            uids, pos, neg = (t.long().to(device) for t in (uids, pos, neg))
            iids = torch.cat([pos, neg], dim=0)
            optimizer.zero_grad()
            loss, _, _ = model(uids, iids, pos, neg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        history['loss'].append(epoch_loss / n_batches)

        if epoch % eval_every == 0 or epoch == n_epochs - 1:
            m = evaluate(model, adj_norm, test_labels, device)
            history['eval_epochs'].append(epoch)
            history['recall20'].append(m['recall@20'])
            history['ndcg20'].append(m['ndcg@20'])

    history['final'] = evaluate(model, adj_norm, test_labels, device)
    return history

In [ ]:
N_EPOCHS = 20  # reduced from the paper's 100 epochs for a fast Colab demo

histories = {}
for mode in ['S0', 'S3']:
    histories[mode] = train_and_evaluate(
        mode, train_coo, n_user, n_item, ppr_pool, deg, test_labels, DEVICE,
        n_epochs=N_EPOCHS)
    print(f"{mode} final: {histories[mode]['final']}")

## 6. Results

In [ ]:
summary = pd.DataFrame({
    mode: {'Recall@20': h['final']['recall@20'], 'NDCG@20': h['final']['ndcg@20']}
    for mode, h in histories.items()
}).T
print(f'({N_EPOCHS}-epoch, single-seed demo -- see results/ablation_summary.csv for the '
      'paper\'s 100-epoch, multi-seed numbers)')
summary

In [ ]:
colors = {'S0': '#1f77b4', 'S3': '#d62728'}
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

for mode, h in histories.items():
    axes[0].plot(h['eval_epochs'], h['recall20'], marker='o', color=colors[mode], label=mode)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Recall@20')
axes[0].set_title(f'Test Recall@20 vs. epoch ({N_EPOCHS}-epoch demo)')
axes[0].legend(frameon=False)

modes = list(histories.keys())
x = np.arange(len(modes))
axes[1].bar(x, [histories[m]['final']['recall@20'] for m in modes],
            color=[colors[m] for m in modes], width=0.5)
axes[1].set_xticks(x); axes[1].set_xticklabels(modes)
axes[1].set_ylabel('Recall@20'); axes[1].set_title('Final Recall@20')

fig.tight_layout()
plt.show()

## 7. Discussion

At this reduced scale (20 epochs, single seed) you should typically see S3 at or
slightly above S0, the same direction as the paper's headline result (Table 2:
S3 = 0.1015 vs. S0 = 0.1005 Recall@20 at 100 epochs, averaged over 3 seeds) -- though
with only one seed and a fifth of the epochs, this single run alone cannot establish
statistical significance; the paper's `results/main_results_summary.csv` and
`results/ablation_summary.csv` report the properly seeded comparison, and
`report/main.tex` §7 discusses why the *further* comparison between S3's degree gate
and the simpler S2 fixed-ramp mixture is a genuine mixed finding (S2 slightly edges
out S3 in the full study) rather than the clear win the degree gate was designed to
produce.

For the full S0/S1/S2/S3 ablation, the degree-bucket (head/mid/tail) breakdown, the
external baselines (MF-BPR, LightGCN, SimGCL), and the hyperparameter-sensitivity
analysis, see `report/main.pdf` and the `results/` directory in the project
repository.